# Checkpointing on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/checkpoints-and-continuous-tuning.md`](../docs/notes/checkpoints-and-continuous-tuning.md)
for the verified SDK surface.

A tuning job can export **one checkpoint per epoch** (up to ~10, evenly spaced
beyond that). Each checkpoint is independently servable, so you can compare them
and pick which one the tuned model serves by default — useful when the last
epoch overfits. This reuses the [SFT](01_sft.ipynb) support-intent dataset and:

1. tunes with `export_last_checkpoint_only=False` (the default → keep every
   checkpoint),
2. lists the checkpoints (`checkpoint_id` / `epoch` / `step` / `endpoint`),
3. runs inference against two checkpoints to show they can diverge, and
4. reads the model's **default checkpoint** and reassigns it.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before running the tune/inference cells.

In [ ]:
from geap_tuning.config import genai_client, load_config

# gemini-2.5-flash supports intermediate checkpoints; tuning stays REGIONAL.
BASE_MODEL = "gemini-2.5-flash"
EPOCHS = 3  # a few epochs -> a few checkpoints to compare
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build and stage the SFT dataset

Same deterministic support-intent splits as the SFT notebook.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.data import build_sft_dataset

paths = build_sft_dataset("../datasets/sft_support_intent")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/checkpoints_sft/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/checkpoints_sft/val.jsonl")
train_uri, val_uri

## 2. Tune, keeping every checkpoint

`export_last_checkpoint_only=False` is the default, but we pass it explicitly to
make the intent obvious. Reuse an existing job with the same display name if one
exists (cost control).

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.sft.tune import launch_sft_job

DISPLAY_NAME = "geap-checkpoints-sft"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        epochs=EPOCHS,
        export_last_checkpoint_only=False,
    )
job = wait_for_tuning_job(client, job.name)
job.state

## 3. List the exported checkpoints

In [ ]:
from geap_tuning.jobs import list_checkpoints

checkpoints = list_checkpoints(job)
for cp in checkpoints:
    print(f"id={cp.checkpoint_id} epoch={cp.epoch} step={cp.step} endpoint={cp.endpoint}")
len(checkpoints)

## 4. Compare two checkpoints on a held-out ticket

Each checkpoint has its own endpoint, so we can call the first and the last on
the same input and see whether they agree.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import checkpoint_endpoint
from geap_tuning.sft.data import SUPPORT_TICKETS, build_records, split_dataset

_, _, test_pairs = split_dataset(SUPPORT_TICKETS)
ticket = build_records(test_pairs[:1])[0]["contents"][0]["parts"][0]["text"]
for cp in (checkpoints[0], checkpoints[-1]):
    reply = generate(client, checkpoint_endpoint(job, cp.checkpoint_id), ticket)
    print(f"[{cp.checkpoint_id}] {reply!r}")

## 5. Read and reassign the default checkpoint

The **default checkpoint** is the one the model's base endpoint serves. If a
later epoch overfit, point the default back at an earlier checkpoint via
`client.models.update(...)` (wrapped by `set_default_checkpoint`).

In [ ]:
from geap_tuning.jobs import get_default_checkpoint_id, set_default_checkpoint

print(f"default before: {get_default_checkpoint_id(client, job)}")
set_default_checkpoint(client, job, checkpoints[0].checkpoint_id)
print(f"default after:  {get_default_checkpoint_id(client, job)}")

## Next steps

Checkpointing pairs naturally with **continuous tuning** — continue-tune from a
specific checkpoint of a prior job. See
[`05_continuous_tuning.ipynb`](05_continuous_tuning.ipynb) for an SFT → RLFT
chain.